In [ ]:
"""
Pontine-tegmentum (PT) control seed: age-group differences in cortical gradient
dispersion during negative movie-viewing (Supplementary Figure 21), and the
direct seed x age test (LC - PT difference score, Young vs Old).

Statistics use group_comparison() from age_dispersion_behaviour_functions.py
unchanged: residualise on covariates, independent-samples t-test, Cohen's d.
Run from the repository root.
"""

import pandas as pd
from age_dispersion_behaviour_functions import group_comparison, fdr_correction

DATA_PATH = 'Source Data/Disperion_source_data/Source_data_LC_Cortex_Gradient.xlsx'
COVARIATES = ['Gender', 'Mean_FD_Negative', 'Cortical_Thickness']

# LC metric : PT metric (right LC, negative movie-viewing)
PAIRS = {
    'Global_Dispersion':          'global_dispersion_PT',
    'Between_network_Dispersion': 'between__dispersion_PT',
    'Dispersion_N6':              'FPN_dispersion_PT',
}

# --- load and merge sheets (PT and Cor_thick sheets are in Demographics row order) ---
xl = pd.ExcelFile(DATA_PATH)
data = (xl.parse('Demographics')
          .merge(xl.parse('Dispersion_Negative'), on='Sub')
          .merge(xl.parse('Motion'), on='Sub'))
data['Cortical_Thickness'] = xl.parse('Cor_thick')['Cortical_Thickness'].values
data[list(PAIRS.values())] = xl.parse('PT_dispersion_sensitivity')[list(PAIRS.values())].values


def report(label, res):
    print(f"  {label:28s} t = {res['t_statistic']:6.2f}  p = {res['p_value']:.3f}  "
          f"d = {abs(res['cohens_d']):.2f}  95% CI [{res['ci_lower']:.3f}, {res['ci_upper']:.3f}]")


# --- 1. PT-seed dispersion: Young vs Old ---
print("PT-seed dispersion, Young vs Old (covariate-adjusted)")
pt_res = {pt: group_comparison(data, pt, COVARIATES) for pt in PAIRS.values()}
_, p_fdr = fdr_correction([r['p_value'] for r in pt_res.values()])
for (pt, r), q in zip(pt_res.items(), p_fdr):
    report(pt, r); print(f"{'':30s} p_FDR = {q:.3f}")

# --- 2. Seed x age interaction: (LC - PT) difference score, Young vs Old (F = t^2) ---
print("\nSeed x age interaction (LC - PT difference score)")
for lc, pt in PAIRS.items():
    data[f'{lc}_minus_PT'] = data[lc] - data[pt]
    r = group_comparison(data, f'{lc}_minus_PT', COVARIATES)
    report(lc, r)
    print(f"{'':30s} F(1, {r['n_group1'] + r['n_group2'] - 2}) = {r['t_statistic']**2:.2f}")